In [13]:
import pandas as pd

In [14]:
df = pd.read_csv("golden_customer_record_dataset.csv", header=1)
df.head()

,record_id,first_name,last_name,email,phone,dob,address
0,rec_2177,Adam,Baker,hhunter@example.net,(390)373-3821x09200,29-06-1958,"68509 Ford Fields, Lake Paul, KS 51738"
1,rec_11443,Kenneth,Sparks,omckinney@example.com,(845)214-7843,24-03-1946,"99103 Vickie Road Suite 101, Kanestad, SD 89185"
2,rec_11551,Margaret,Smith,kmartin@example.com,+1-562-601-5867x65530,31-07-1999,"60241 Sandra Burgs, West Laura, OH 19750"
3,rec_769,Carrie,Mcclain,fordbeverly@example.org,998-812-5597x77265,08-09-1959,"580 Johnson Cliffs, Davidmouth, CT 51295"
4,rec_3070,Luis,Herman,amandabrown@example.net,(526)533-6538x4777,29-01-1984,"2528 Coleman Alley, Scottmouth, IL 20897"


In [15]:
df.columns

Index(['record_id', 'first_name', 'last_name', 'email', 'phone', 'dob',
       'address'],
      dtype='str')

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   record_id   50000 non-null  str  
 1   first_name  50000 non-null  str  
 2   last_name   50000 non-null  str  
 3   email       50000 non-null  str  
 4   phone       50000 non-null  str  
 5   dob         50000 non-null  str  
 6   address     50000 non-null  str  
dtypes: str(7)
memory usage: 2.7 MB


In [17]:
df.isnull().sum()

record_id     0
first_name    0
last_name     0
email         0
phone         0
dob           0
address       0
dtype: int64

#Data Exploration

The dataset was loaded using pandas and explored to understand its structure. Key fields such as customer name, email, phone number, and address were identified as important attributes for resolving duplicate customer records.
The dataset contains no missing values.

#Field Selection for Entity Resolution

The following fields were identified as key attributes for resolving customer entities:

Email:
A unique identifier for customers; exact matches strongly indicate the same individual.
Phone Number:
Another strong identifier, as it is typically unique to a customer.
Full Name (First + Last Name):
Used for fuzzy matching to identify similar customer names with minor variations.
Date of Birth (DOB):
Helps validate identity when combined with name or other attributes.
Address:
Used for fuzzy matching, especially when names are similar.

In [18]:
df["full_name"] = df["first_name"] + " " + df["last_name"]

In [19]:
# data cleaning
df["email"] = df["email"].str.lower().str.strip()
df["phone"] = df["phone"].str.replace(r"\D", "", regex=True)
df["full_name"] = df["full_name"].str.lower().str.strip()
df["address"] = df["address"].str.lower().str.strip()

In [20]:
from fuzzywuzzy import fuzz

In [21]:
def is_match(row1, row2):
    # Rule 1: Exact match
    if row1['email'] == row2['email']:
        return True
    if row1['phone'] == row2['phone']:
        return True

    # Rule 2: Name + DOB
    if row1['full_name'] == row2['full_name'] and row1['dob'] == row2['dob']:
        return True

    # Rule 3: Fuzzy match
    name_score = fuzz.token_sort_ratio(row1['full_name'], row2['full_name'])
    address_score = fuzz.token_sort_ratio(row1['address'], row2['address'])

    if name_score >= 90 and address_score >= 80:
        return True

    return False

#Entity Resolution Logic

Customer records were matched using a combination of deterministic and fuzzy matching rules:

Exact matches were performed using email and phone number.
Semi-strong matches were identified using full name and date of birth.
Fuzzy matching techniques were applied on names and addresses using similarity thresholds to capture minor variations.

This multi-rule approach ensures both precision and flexibility in identifying duplicate customer records.

In [22]:
# block key
df["block_key"] = (
    df["email"].fillna("") + "_" +
    df["phone"].fillna("") + "_" +
    df["full_name"].str[0] + "_" +
    df["dob"]
)
# initialization
df["cluster_id"] = -1
cluster_id = 0

In [23]:
# clustering logic
for block, group in df.groupby("block_key"):
    indices = group.index.tolist()

    for i in range(len(indices)):
        idx_i = indices[i]

        if df.at[idx_i, "cluster_id"] == -1:
            df.at[idx_i, "cluster_id"] = cluster_id

            for j in range(i + 1, len(indices)):
                idx_j = indices[j]

                if df.at[idx_j, "cluster_id"] == -1:
                    if is_match(df.loc[idx_i], df.loc[idx_j]):
                        df.at[idx_j, "cluster_id"] = cluster_id

            cluster_id += 1

#Cluster Assignment

A clustering approach was implemented to group records belonging to the same customer. Each record was assigned a unique cluster_id based on matching rules. Records satisfying the defined matching conditions were grouped under the same cluster.

#Optimization using Blocking

To improve performance, a blocking technique was applied to reduce the number of comparisons. Records were grouped using a composite blocking key based on email, phone, name, and date of birth.

Matching was then performed only within each block, significantly improving computational efficiency while maintaining accuracy.

In [24]:
df.to_csv("gcr_output.csv", index=False)

In [25]:
df["cluster_id"].nunique()

45782